# Week 4 Live Coding: Does This Message Move Voters?

Data: a survey experiment with 5,000 respondents testing 4 persuasion messages.

Three tasks:
1. Compute the ATE for each message arm vs. control
2. Run randomization inference on the "winning" arm
3. Simulate the multiple-comparisons problem


### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk04_message_testing/data/survey_experiment.csv')
df.shape

In [ ]:
df.head()

## Part 1: ATEs for each message arm

Compute the mean of `vote_for_candidate` for each arm. The difference between each treatment arm and the control arm is the estimated ATE.


In [ ]:
# Mean vote intention by arm
df.groupby('arm')['vote_for_candidate'].mean()

Now compute the difference from control for each treatment arm.


In [ ]:
control_mean = df[df['arm'] == 'control']['vote_for_candidate'].mean()
print(f'Control mean: {control_mean:.3f}')
print()

for arm in ['economy', 'healthcare', 'character', 'abortion']:
    arm_mean = df[df['arm'] == arm]['vote_for_candidate'].mean()
    diff = arm_mean - control_mean
    print(f'{arm}: {arm_mean:.3f}  (diff from control: {diff:+.4f})')

The abortion arm shows the largest difference: about +4.9 percentage points above control. The other three are small and mixed in sign.

Is the +4.9 difference real, or is it noise?


## Part 2: Randomization inference on the "winning" arm

We will use the same procedure from last week. Subset to just the control and abortion arms, then shuffle the treatment labels 1,000 times.


**Step 1: One fake re-assignment.**

We take the control + abortion subset, shuffle the arm labels, and compute a fake ATE.


In [ ]:
# Subset to control and abortion arms only
subset = df[df['arm'].isin(['control', 'abortion'])].copy()
print(f'Rows in subset: {len(subset)}')
print(f'Abortion arm: {(subset["arm"] == "abortion").sum()}')
print(f'Control arm: {(subset["arm"] == "control").sum()}')

In [ ]:
# The observed ATE
observed_ate = (
    subset[subset['arm'] == 'abortion']['vote_for_candidate'].mean()
    - subset[subset['arm'] == 'control']['vote_for_candidate'].mean()
)
print(f'Observed ATE: {observed_ate:.4f}')

In [ ]:
# One fake re-assignment
np.random.seed(2026)
# Same .sample(frac=1).values pattern as W3 — shuffle the labels, keep outcomes fixed.
# The only difference: W3 shuffled 0/1 treatment labels; here we shuffle 'abortion'/'control' arm labels.
shuffled_labels = subset['arm'].sample(frac=1).values
fake_treated = subset['vote_for_candidate'][shuffled_labels == 'abortion']
fake_control = subset['vote_for_candidate'][shuffled_labels == 'control']
fake_ate = fake_treated.mean() - fake_control.mean()
print(f'Fake ATE from one shuffle: {fake_ate:.4f}')

**Step 2: Now do that 1,000 times.**

We repeat the shuffle 1,000 times and store each fake ATE. This builds the null distribution: what would the ATE look like if the abortion message had no effect?


In [ ]:
# What this loop does in plain English:
# 1. Shuffle the arm labels (so the "abortion" and "control" tags are randomly reassigned)
# 2. Compute the difference in means under this fake assignment
# 3. Store that fake ATE
# 4. Repeat 1,000 times

np.random.seed(2026)
fake_ates = []

for i in range(1000):
    # Same shuffle as W3: .sample(frac=1).values randomizes the labels.
    shuffled_labels = subset['arm'].sample(frac=1).values
    fake_treated = subset['vote_for_candidate'][shuffled_labels == 'abortion']
    fake_control = subset['vote_for_candidate'][shuffled_labels == 'control']
    fake_ate = fake_treated.mean() - fake_control.mean()
    fake_ates.append(fake_ate)

In [ ]:
# p-value: how often does a RANDOM reshuffle produce a fake gap at least as far
# FROM ZERO as the one we really saw? We take abs() of both because a big *negative*
# fake gap (say -5 pp) is just as surprising as a big *positive* one -- "no effect"
# predicts a gap near zero, so "extreme" means far from zero in either direction.
# (Two-sided -- same convention as W3.)
p_value = np.mean([abs(f) >= abs(observed_ate) for f in fake_ates])
print(f'Observed ATE: {observed_ate:.4f}')
print(f'Two-sided p-value: {p_value:.3f}')

In isolation, this p-value suggests the abortion arm's effect is unlikely to be pure chance.

But we did not test the abortion arm in isolation. We tested four messages and picked the winner. That changes the calculation.


## Part 3: The multiple-comparisons simulation

Suppose all four messages have zero effect. We simulate 1,000 fake experiments and ask: how often does at least one arm look significant?


First: what counts as "significant"? We already built a null distribution in Part 2. The 95th percentile of the absolute fake ATEs gives us a threshold: any difference larger than this would be in the most extreme 5% under the null. (This is the same idea as the p-value, flipped around: instead of asking "what fraction of fakes beat the real gap?", we find the cutoff that only the top 5% of fakes clear.)


In [ ]:
# Derive the significance threshold from the RI null distribution
threshold = np.percentile([abs(f) for f in fake_ates], 95)
print(f'95th percentile of |fake ATEs|: {threshold:.4f}')
print(f'Any ATE larger than {threshold:.4f} in absolute value would be "significant" at the 5% level.')

**Step 1: One fake experiment.**

Generate 5,000 outcomes with no treatment effect. Assign to 5 arms. Compute the 4 differences from control. Check if any cross the threshold.


In [ ]:
# One fake experiment: no treatment effect
np.random.seed(2026)

# Generate 5000 outcomes from the same distribution (no effect)
fake_votes = np.random.binomial(1, 0.44, size=5000)

# First 1000 = control, next 4 groups of 1000 = treatment arms
fake_control_mean = fake_votes[:1000].mean()

# Check each treatment arm
any_significant = False
for j in range(4):
    arm_name = ['economy', 'healthcare', 'character', 'abortion'][j]
    arm_start = 1000 * (j + 1)
    arm_end = arm_start + 1000
    arm_mean = fake_votes[arm_start:arm_end].mean()
    diff = abs(arm_mean - fake_control_mean)
    sig = '***' if diff >= threshold else ''
    print(f'{arm_name}: diff = {diff:.4f} {sig}')
    if diff >= threshold:
        any_significant = True

print(f'\nAt least one significant? {any_significant}')

**Step 2: Now do that 1,000 times.**

We repeat the fake experiment 1,000 times and track how often at least one arm crosses the threshold.


In [ ]:
# What this loop does in plain English:
# 1. Generate 5,000 fake outcomes where no message has any effect
# 2. Split into 5 groups of 1,000 (one control, four treatment arms)
# 3. Check if ANY of the 4 treatment arms has a difference from control
#    larger than our significance threshold
# 4. Record whether this fake experiment produced a "winner"
# 5. Repeat 1,000 times

np.random.seed(2026)
false_positive_count = 0

for i in range(1000):
    fake_votes = np.random.binomial(1, 0.44, size=5000)
    fake_control_mean = fake_votes[:1000].mean()
    
    any_sig = False
    for j in range(4):
        arm_mean = fake_votes[1000*(j+1) : 1000*(j+2)].mean()
        if abs(arm_mean - fake_control_mean) >= threshold:
            any_sig = True
            break
    
    if any_sig:
        false_positive_count += 1

print(f'Out of 1,000 fake experiments:')
print(f'At least one arm looked significant: {false_positive_count} times')
print(f'False positive rate: {false_positive_count / 1000:.3f}')

**Stop and think about what this means for the consultant's claim.**

About 1 in 7 fake experiments (\~15% here) produce a "winner" even when no message does anything. The consultant ran one real experiment with four arms and found one that crossed the threshold.

(Don't worry that it isn't exactly the 18.5% on the slide. You didn't break anything. That formula assumes the four tests are completely separate; here all four arms are compared against the *same* control group, which nudges the number down a little. Either way it happens far more often than the 1-in-20 you might naively expect.)

Her result is consistent with pure noise. That does not mean Message D definitely has no effect. It means we cannot distinguish the result from what chance alone would produce.

This is the multiple-comparisons problem: the more things you test, the more likely you are to find something "significant" by luck.


## Optional: visualize the null distribution


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(fake_ates, bins=40, edgecolor='white', alpha=0.7)
plt.axvline(observed_ate, color='red', linewidth=2, label=f'Observed ATE = {observed_ate:.4f}')
plt.xlabel('Fake ATE (abortion arm vs. control)')
plt.ylabel('Count')
plt.title('Null distribution from 1,000 re-randomizations')
plt.legend()
plt.tight_layout()
plt.show()